# Build Team Dataset with Per-Team Batting Stats

Parameterized notebook for building stadium-specific datasets with **separated home/away batting statistics** and weather data.

**To switch teams**: Change `TEAM` below and re-run all cells.  
**To rebuild all teams**: Set `RUN_ALL_TEAMS = True`.

### Prerequisites
Run `master_data_pull.ipynb` first — it enriches `master_data.csv` with per-team batting columns from Statcast.

### Pipeline (per team)
1. Load enriched `master_data.csv` and filter to home games for selected team/seasons
2. Fetch weather from Open-Meteo (lat/lon/timezone from `team_parameters.csv`)
3. Compute wind projections onto outfield vectors (CF, LCF, RCF bearings from `team_parameters.csv`)
4. Merge and save team CSV (51 columns)

In [ ]:
# ============================================================
# PARAMETERS — change these to switch teams
# ============================================================
TEAM = 'SF'           # <- change this (see team_parameters.csv for all codes)
RUN_ALL_TEAMS = False  # <- set True to rebuild all 30 teams

# Season range auto-loaded from team_parameters.csv.
# Uncomment below to override:
# SEASON_START = 2021
# SEASON_END = 2025

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

# --- Load team parameters ---
params_df = pd.read_csv(os.path.join('..', 'analysis', 'team_parameters.csv'))

# --- Load enriched master data ---
master = pd.read_csv('master_data.csv')
master['game_date'] = pd.to_datetime(master['game_date'])

# Verify master has per-team batting columns
bat_cols = [c for c in master.columns if c.startswith('home_bat_') or c.startswith('away_bat_')]
assert len(bat_cols) >= 16, (
    f"Expected at least 16 per-team batting columns, found {len(bat_cols)}. "
    "Run master_data_pull.ipynb first!"
)

print(f"Master dataset: {len(master)} games, {len(master.columns)} columns")
print(f"Per-team batting columns: {len(bat_cols)}")

# --- Determine which teams to process ---
if RUN_ALL_TEAMS:
    teams_to_process = params_df['team_code'].tolist()
else:
    teams_to_process = [TEAM]

print(f"\nTeams to process: {teams_to_process}")

## Section 1: Weather & Wind Functions

Reused from `giants_analysis.ipynb` — Open-Meteo fetch, 3-hour game averaging, wind projection onto outfield vectors.

In [ ]:
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"


def fetch_season_weather(lat, lon, timezone, year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': lat,
        'longitude': lon,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': timezone,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']

    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))

    return df


def get_game_weather(game_start_dt, hourly_df):
    """Average weather over the 3 hours following game start."""
    start = game_start_dt
    end = start + timedelta(hours=2)

    window = hourly_df.loc[start:end]

    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })

    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),
        'wspd': window['wspd'].mean(),
    }

    # Circular mean for wind direction
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan

    return pd.Series(result)


def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]


def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees)
    Returns: positive = blowing OUT, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    wind_toward = (wdir + 180) % 360
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))


print("Weather and wind functions defined.")

## Section 2: Build Team Dataset Function

Core function that takes a team code, filters master data, fetches weather, computes wind projections, and saves the final CSV.

In [ ]:
def build_team_dataset(team_code, params_df, master):
    """
    Build a complete team dataset with weather + wind projections.
    
    Parameters
    ----------
    team_code : str
        Team code (e.g. 'SF', 'BOS')
    params_df : DataFrame
        Team parameters with lat, lon, timezone, bearings
    master : DataFrame
        Enriched master_data with per-team batting columns
    
    Returns
    -------
    DataFrame with ~47 columns, or None on failure
    """
    # --- Team config ---
    team_row = params_df[params_df['team_code'] == team_code]
    if len(team_row) == 0:
        print(f"  ERROR: Team '{team_code}' not found in team_parameters.csv")
        return None
    team_row = team_row.iloc[0]

    team_name = team_row['team_name']
    stadium = team_row['stadium_name']
    dataset_file = team_row['dataset_file']
    start_year = int(team_row['data_start_year'])
    end_year = int(team_row['data_end_year'])
    lat = team_row['latitude']
    lon = team_row['longitude']
    tz = team_row['timezone']
    cf_dir = float(team_row['cf_bearing'])
    lcf_dir = float(team_row['lcf_bearing'])
    rcf_dir = float(team_row['rcf_bearing'])

    seasons = range(start_year, end_year + 1)

    print(f"\n{'='*60}")
    print(f"  {stadium} ({team_name}) — {team_code}")
    print(f"  Seasons: {start_year}-{end_year}")
    print(f"  Output: {dataset_file}")
    print(f"{'='*60}")

    # --- Filter master to home games ---
    games = master[
        (master['home_team'] == team_code) &
        (master['season'].isin(seasons))
    ].copy()

    if len(games) == 0:
        print(f"  WARNING: No games found for {team_code}")
        return None

    # Convert game_start_utc to local timezone
    games['game_start'] = (
        pd.to_datetime(games['game_start_utc'], utc=True)
        .dt.tz_convert(tz)
        .dt.tz_localize(None)
    )
    games['start_hour'] = games['game_start'].dt.hour
    games['game_date'] = pd.to_datetime(games['game_date'])
    games = games.drop(columns=['home_team', 'game_start_utc'])
    games = games.sort_values('game_date').reset_index(drop=True)

    print(f"  Games: {len(games)}")

    # --- Fetch weather season by season ---
    weather_records = []
    for year in seasons:
        try:
            hourly_df = fetch_season_weather(lat, lon, tz, year)
        except Exception as e:
            print(f"  WARNING: Weather fetch failed for {year}: {e}")
            season_games = games[games['season'] == year]
            for _, game in season_games.iterrows():
                weather_records.append({
                    'game_pk': game['game_pk'],
                    'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                    'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
                })
            continue

        season_games = games[games['season'] == year]
        for _, game in season_games.iterrows():
            if pd.isna(game['game_start']):
                weather_records.append({
                    'game_pk': game['game_pk'],
                    'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                    'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
                })
                continue

            game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
            wx = get_game_weather(game_hour, hourly_df)
            wx['game_pk'] = game['game_pk']
            weather_records.append(wx.to_dict())

    weather_df = pd.DataFrame(weather_records)
    weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')

    # --- Merge weather ---
    games_full = games.merge(weather_df, on='game_pk', how='left')

    # --- Wind direction bucketing ---
    games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

    # --- Wind projections ---
    games_full['wind_cf'] = games_full.apply(
        lambda r: compute_wind_projection(r['wdir'], r['wspd'], cf_dir), axis=1)
    games_full['wind_lcf'] = games_full.apply(
        lambda r: compute_wind_projection(r['wdir'], r['wspd'], lcf_dir), axis=1)
    games_full['wind_rcf'] = games_full.apply(
        lambda r: compute_wind_projection(r['wdir'], r['wspd'], rcf_dir), axis=1)

    # --- Unit conversions ---
    games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
    games_full['wspd_mph'] = games_full['wspd'] * 0.621371

    # --- Final column order (47 columns, no barrel stats) ---
    final_columns = [
        # Game identification
        'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
        # Scoring
        'home_runs_scored', 'away_runs_scored', 'total_runs',
        # Combined batting stats
        'home_runs_hit', 'strikeouts', 'walks', 'hits',
        'total_pitches', 'avg_exit_velocity',
        'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
        # Home team batting
        'home_bat_hr', 'home_bat_k', 'home_bat_bb', 'home_bat_h',
        'home_bat_pitches', 'home_bat_exit_velo',
        'home_bat_bbe', 'home_bat_hr_h_ratio',
        # Away team batting
        'away_bat_hr', 'away_bat_k', 'away_bat_bb', 'away_bat_h',
        'away_bat_pitches', 'away_bat_exit_velo',
        'away_bat_bbe', 'away_bat_hr_h_ratio',
        # Weather
        'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
        'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
        # Wind projections
        'wind_cf', 'wind_lcf', 'wind_rcf',
    ]

    # Only include columns that exist
    available = [c for c in final_columns if c in games_full.columns]
    result = games_full[available].copy()
    result = result.sort_values('game_date').reset_index(drop=True)

    # Round floating point columns
    round_map = {
        'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
        'home_bat_exit_velo': 1, 'away_bat_exit_velo': 1,
        'home_bat_hr_h_ratio': 4, 'away_bat_hr_h_ratio': 4,
        'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
        'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
        'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
    }
    for col, decimals in round_map.items():
        if col in result.columns:
            result[col] = result[col].round(decimals)

    print(f"  Final: {result.shape[0]} rows x {result.shape[1]} columns")
    missing_cols = set(final_columns) - set(available)
    if missing_cols:
        print(f"  WARNING: Missing columns: {missing_cols}")

    return result


print("build_team_dataset() defined.")

## Section 3: Run Pipeline & Save

Process each team, validate, and save to the data directory.

In [ ]:
results_summary = []

for team_code in teams_to_process:
    team_row = params_df[params_df['team_code'] == team_code].iloc[0]
    dataset_file = team_row['dataset_file']

    team_df = build_team_dataset(team_code, params_df, master)

    if team_df is None:
        results_summary.append({'team': team_code, 'status': 'FAILED', 'rows': 0, 'cols': 0})
        continue

    # Save
    output_path = os.path.join('.', dataset_file)
    team_df.to_csv(output_path, index=False)

    # Quick validation
    n_rows = len(team_df)
    n_cols = len(team_df.columns)
    null_pct = team_df[[c for c in team_df.columns if 'bat_' in c]].isna().mean().mean() * 100

    # Strikeouts check
    if 'home_bat_k' in team_df.columns and 'away_bat_k' in team_df.columns:
        valid = team_df.dropna(subset=['home_bat_k', 'away_bat_k'])
        k_close = (abs(valid['home_bat_k'] + valid['away_bat_k'] - valid['strikeouts']) <= 2).mean() * 100
    else:
        k_close = 0

    results_summary.append({
        'team': team_code,
        'status': 'OK',
        'rows': n_rows,
        'cols': n_cols,
        'batting_null_pct': round(null_pct, 1),
        'k_validation_pct': round(k_close, 1),
        'file': dataset_file,
    })

    print(f"  Saved: {dataset_file} ({n_rows} rows, {n_cols} cols)")
    print(f"  Batting null rate: {null_pct:.1f}%, K validation: {k_close:.1f}%")

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))

In [ ]:
# --- Detailed validation for most recent team ---
if team_df is not None:
    print(f"Detailed validation for {teams_to_process[-1]}:")
    print(f"\n--- Column check ---")
    print(f"  Columns: {len(team_df.columns)} (expected ~47)")
    print(f"  Column list: {team_df.columns.tolist()}")

    print(f"\n--- Null counts ---")
    key_cols = [c for c in team_df.columns if 'bat_' in c or c in ['temp_f', 'wspd', 'wind_cf']]
    for col in key_cols:
        n_null = team_df[col].isna().sum()
        pct = 100 * n_null / len(team_df)
        status = "OK" if pct < 5 else "WARNING"
        print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

    print(f"\n--- Cross-validation ---")
    valid = team_df.dropna(subset=['home_bat_k', 'away_bat_k', 'home_bat_hr', 'away_bat_hr'])

    # K check
    k_diff = abs(valid['home_bat_k'] + valid['away_bat_k'] - valid['strikeouts'])
    print(f"  K: sum within ±2 of combined: {(k_diff <= 2).mean()*100:.1f}%")
    print(f"      mean diff: {k_diff.mean():.2f}, max diff: {k_diff.max():.0f}")

    # HR check
    hr_diff = abs(valid['home_bat_hr'] + valid['away_bat_hr'] - valid['home_runs_hit'])
    print(f"  HR: sum within ±1 of combined: {(hr_diff <= 1).mean()*100:.1f}%")
    print(f"      mean diff: {hr_diff.mean():.2f}, max diff: {hr_diff.max():.0f}")

    print(f"\n--- Summary stats (per-team batting) ---")
    bat_cols = [c for c in team_df.columns if c.startswith('home_bat_') or c.startswith('away_bat_')]
    print(team_df[bat_cols].describe().round(3).T[['mean', 'std', 'min', 'max']].to_string())

In [ ]:
# ============================================================
# PARAMETERS — change these to switch teams
# ============================================================
TEAM = 'SF'           # <- change this (see team_parameters.csv for all codes)
RUN_ALL_TEAMS = False  # <- set True to rebuild all 30 teams

# Season range auto-loaded from team_parameters.csv.
# Uncomment below to override:
# SEASON_START = 2021
# SEASON_END = 2025

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import time
import requests
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

# --- Load team parameters ---
params_df = pd.read_csv(os.path.join('..', 'analysis', 'team_parameters.csv'))

# --- Load master data once ---
master = pd.read_csv('master_data.csv')
master['game_date'] = pd.to_datetime(master['game_date'])
print(f"Master dataset: {len(master)} total games")
print(f"Teams in params: {len(params_df)}")

# --- Determine which teams to process ---
if RUN_ALL_TEAMS:
    teams_to_process = params_df['team_code'].tolist()
else:
    teams_to_process = [TEAM]

print(f"\nTeams to process: {teams_to_process}")

## Section 1: Statcast Per-Team Batting Stats

Fetch pitch-level data via `pybaseball.statcast()` in 2-week chunks, filter to relevant game_pks, separate by `inning_topbot` (Top = away batting, Bot = home batting), and aggregate per game per team.

Cached to `data/statcast_cache/*.parquet` so re-runs skip already-downloaded date ranges.